# Model 3: RAG — FAISS + Qwen2.5-1.5B (lokal, kein API)
**Modell:** `Qwen2.5-1.5B-Instruct` (lokal)  
**Methode:** Retrieval-Augmented Generation — relevante Paragraphen aus EStG, KStG, UStG, BAO, GrEStG werden semantisch abgerufen und als Kontext mitgegeben


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "sentence-transformers", "faiss-cpu", "tqdm"])
print("✓ Pakete bereit")

In [ ]:
import os, csv
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

DATASET     = "../../dataset_clean.csv"
OUT         = "../results/results_model3_rag.csv"
LEGAL_DIR   = "legal_texts"
INDEX_CACHE = "rag_index.npz"
os.makedirs("../results", exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

def load_questions():
    with open(DATASET, newline="", encoding="utf-8-sig") as f:
        return [{"id": r["id"], "prompt": r["prompt"]} for r in csv.DictReader(f)]

questions = load_questions()
print(f"{len(questions)} Fragen geladen.")

In [ ]:
# ── FAISS Index aufbauen ────────────────────────────────
CHUNK_SIZE, OVERLAP, TOP_K = 600, 100, 4
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def chunk_text(text, source):
    chunks, i = [], 0
    while i < len(text):
        chunks.append({"text": text[i:i+CHUNK_SIZE], "source": source})
        i += CHUNK_SIZE - OVERLAP
    return chunks

all_chunks = []
for fname in sorted(os.listdir(LEGAL_DIR)):
    if fname.endswith(".txt"):
        with open(f"{LEGAL_DIR}/{fname}", encoding="utf-8") as f:
            cks = chunk_text(f.read(), fname.replace(".txt",""))
        all_chunks.extend(cks)
        print(f"  {fname}: {len(cks)} Chunks")
print(f"Gesamt: {len(all_chunks)} Chunks")

print("\nLade Embedder...")
embedder = SentenceTransformer(EMBED_MODEL)

if os.path.exists(INDEX_CACHE):
    embeddings = np.load(INDEX_CACHE, allow_pickle=True)["embeddings"]
    print("Index aus Cache geladen.")
else:
    print("Embedde Chunks (~1 Min.)...")
    embeddings = embedder.encode([c["text"] for c in all_chunks],
                                  batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.savez(INDEX_CACHE, embeddings=embeddings)

faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings.astype(np.float32))
print(f"✓ FAISS Index: {faiss_index.ntotal} Vektoren")

In [ ]:
# ── Sprachmodell laden ──────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Lade {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
model.eval()
print("✓ Modell geladen")

In [ ]:
SYSTEM = (
    "Du bist ein Experte für österreichisches Steuerrecht. "
    "Beantworte die Frage auf Basis der bereitgestellten Gesetzestexte. "
    "Sei präzise und beziehe dich auf die relevanten Paragraphen."
)

def retrieve(query):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype(np.float32)
    _, idx = faiss_index.search(q_emb, TOP_K)
    return [all_chunks[i] for i in idx[0]]

def infer_rag(question):
    context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in retrieve(question))
    full_q = f"Gesetzestexte:\n{context}\n\nFrage: {question}"
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": full_q}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.1,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Test
print(infer_rag("Was ist der Grunderwerbsteuersatz beim Kauf zwischen Fremden?"))

In [ ]:
results = []
for q in tqdm(questions, desc="Model 3 RAG"):
    results.append({"id": q["id"], "answer": infer_rag(q["prompt"])})

df = pd.DataFrame(results)
df.to_csv(OUT, index=False)
print(f"✓ Gespeichert: {OUT}  ({len(df)} Zeilen)")
df.head(3)